# P300 Classifiers

### 1. Reshaping, epoching & Visualizing Imbalance

Code to filter and classify the data was adapted from the following repository (with the dataset creators' own scripts to handle and classify the data): https://github.com/plcrodrigues/py.BI.EEG.2015a-GIPSA <br>
More specifically, I adapted the code from the evoked_potentials.py and classification_scores.py files, where I got the oarameters and the epoching/filtering sample code with MNE.

Loading the data:

In [ ]:
from project_config import EEG_DATA_DIR
import pandas as pd
header_file_path = str(EEG_DATA_DIR / 'Header.csv')
header_df = pd.read_csv(header_file_path)
header_df.head()

In [ ]:
# usingn one session to explore the data
session_file_path = str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv')
session_df = pd.read_csv(session_file_path, header=None, names=header_df.columns)
session_df.head()

Calculate session duration:

In [ ]:
total_duration_seconds = session_df.iloc[-1, 0] - session_df.iloc[0, 0]
total_duration_minutes = total_duration_seconds / 60
print ( total_duration_minutes, "minutes long")

Reshaping data, epoch extraction and filtering 1-24Hz with firwin:

In [ ]:
import mne 
import numpy as np
header_path = str(EEG_DATA_DIR / 'Header.csv')
data_path = str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv')
header = pd.read_csv(header_path)
data = pd.read_csv(data_path, names=header.columns)
data = data.dropna()
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin") 
# firwin for noise reduction,
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
X = epochs.get_data()
y = epochs.events[:, -1] - 1

Exploring 'Target' imbalance:

In [ ]:
header_path = str(EEG_DATA_DIR / 'Header.csv')
session_data_path = str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv')
header_df = pd.read_csv(header_path)
column_names = header_df.columns.tolist()
session_df = pd.read_csv(session_data_path, names=column_names)
target_counts = session_df["Target"].value_counts(normalize=True)
target_counts

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
target_counts.plot(kind="bar")
plt.title("Distribution of Target variable")
plt.xlabel("value")
plt.ylabel("frequency")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

Checking ruke of thumb (features:data points ratio = 1:10):

In [ ]:
# rows (data points)
num_data_points = session_df.shape[0]
num_features = session_df.shape[1] - 3  
required_data_points = num_features * 10
meets_rule_of_thumb = num_data_points >= required_data_points
num_features, num_data_points, required_data_points, meets_rule_of_thumb

How the data split was chosen: 64-16-20 (train-val-holdout test), as described here: https://www.datarobot.com/wiki/training-validation-holdout/ <br>
Learning curve code was adapted from: https://scikit-learn.org/stable/auto_examples/model_selection/plot_learning_curve.html <br>
Information about firwin filter utilised: https://mne.tools/dev/auto_tutorials/preprocessing/25_background_filtering.html

Importing modules:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, ShuffleSplit, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, f1_score, roc_auc_score, roc_curve
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as IMBPipeline
from imblearn.pipeline import make_pipeline as make_pipeline_imb
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mne.epochs import Epochs
from pyriemann.estimation import ERPCovariances
from pyriemann.classification import MDM
import mne
import warnings

## 2. SVM

Holdout sets were used to evaluate the models' robustness based on the following information: https://www.datarobot.com/wiki/training-validation-holdout/ <br>
The roc-auc plotting code was taken from sklearn's documentation: https://scikit-learn.org/1.1/auto_examples/model_selection/plot_roc.html <br>
--> These are the specific lines: <br>
plt.plot([0, 1], [0, 1], color="navy", lw=lw, linestyle="--") <br>
plt.xlim([0.0, 1.0]) <br>
plt.ylim([0.0, 1.05]) <br>
plt.xlabel("False Positive Rate") <br>
plt.ylabel("True Positive Rate") <br>
plt.title("Receiver operating characteristic example") <br>
plt.legend(loc="lower right") <br>
plt.show()<br>

Firstly training s baseline model for susequent comparison (all predictions evaluate to Non-Target):

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)
header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),

]
header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)

eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
X = epochs.get_data() # features
y = epochs.events[:, -1] - 1 # labels
X_flat = X.reshape(X.shape[0], -1)
# training and holdout sets
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
y_baseline_pred = np.zeros_like(y_holdout) # baseline<. everything as non target
y_baseline_pred_proba = np.zeros_like(y_holdout, dtype=float) + 1 # prob of non-target is 1 (all are non target)

print("classification report for baseline model - no predictions:\n")
print(classification_report(y_holdout, y_baseline_pred, target_names=["Non-Target", "Target"]))
roc_auc_baseline = roc_auc_score(y_holdout, y_baseline_pred_proba)
print(f"baseline model ROC AUC: {roc_auc_baseline:.4f}")
fpr_base, tpr_base, _ = roc_curve(y_holdout, y_baseline_pred_proba, pos_label=1)
plt.figure(figsize=(6, 6))
plt.plot(fpr_base, tpr_base, color="blue", lw=2, label=f'Baseline ROC curve (AUC = {roc_auc_baseline:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic - Baseline Model')
plt.legend(loc="lower right")
plt.show()

Here for the k-fold loop, I found some useful lines of code (specifically 156-1569) here, but i kept getting errors so ChatGPT helped in adapting them for my code: https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/model_selection/_split.py#L1000. <br>
They were adapted to the current k-fold loop I have, right under the train-tets split.

Evaluation on a holdout set:

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)
header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
X = epochs.get_data()
y = epochs.events[:, -1] - 1

X_flat = X.reshape(X.shape[0], -1)

# split into training+validation and holdout sets first (80% train+val, 20% holdout)
X_train_val, X_holdout, y_train_val, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
# split the training+validation set into training (80% of train+val which is 64% of original) and validation (20% of train+val which is 16% of original)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # kfold on training set
roc_auc_scores = []
target_f1_scores = []

for train_index, test_index in skf.split(X_train, y_train):
    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
    y_train_fold, y_test_fold = y_train[train_index], y_train[test_index]
    n_minority_samples = min(np.bincount(y_train_fold))
    k_neighbors = max(1, min(5, n_minority_samples - 1))
    pipeline = IMBPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)),
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="linear", probability=True, random_state=42))
    ])
    pipeline.fit(X_train_fold, y_train_fold)
    y_pred = pipeline.predict(X_test_fold)
    y_pred_proba = pipeline.predict_proba(X_test_fold)[:, 1]
    roc_auc = roc_auc_score(y_test_fold, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test_fold, y_pred, pos_label=1)
    target_f1_scores.append(f1)
pipeline.fit(X_train_val, y_train_val) # fit pipeline on the training set (train + val) before evaluating on holdout 
y_holdout_pred = pipeline.predict(X_holdout)
y_holdout_pred_proba = pipeline.predict_proba(X_holdout)[:, 1]

print("classification report on holdout set - no baseline:\n", classification_report(y_holdout, y_holdout_pred, target_names=["Non-Target", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_holdout_pred_proba)
print(f"holdout set roc auc: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"roc curve (auc = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

With PCA (scores are lower so not mentioned in the thesis report):

In [ ]:
import pandas as pd
import numpy as np
import mne
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, f1_score
from imblearn.pipeline import Pipeline as IMBPipeline
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_05_csv/subject_05_session_03.csv')
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
X = epochs.get_data()
y = epochs.events[:, -1] - 1
X_flat = X.reshape(X.shape[0], -1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipeline = IMBPipeline([
    ("smote", SMOTE(random_state=42)), # balancing data
    ("scaler", StandardScaler()),     # normalization
    ("pca", PCA(n_components=0.95)),  # PCA keeping 95% of variance
    ("svm", SVC(kernel="linear", probability=True, random_state=42))
])

roc_auc_scores = []
target_f1_scores = []
for train_index, test_index in skf.split(X_flat, y):
    X_train, X_test = X_flat[train_index], X_flat[test_index]
    y_train, y_test = y[train_index], y[test_index]
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    target_f1_scores.append(f1)
    
    print("classification report for fold:\n", classification_report(y_test, y_pred, target_names=["non-target", "target"]))
average_roc_auc = np.mean(roc_auc_scores)
average_target_f1 = np.mean(target_f1_scores)
print(f"average roc auc across folds: {average_roc_auc:.4f}")
print(f"average f1 score for the target class: {average_target_f1:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_test, y_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"roc curve (auc = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - last fold")
plt.legend(loc="lower right")
plt.show()

Adding simple baseline correction:

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)

header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True, baseline=(-0.2, 0))
X = epochs.get_data()
y = epochs.events[:, -1] - 1
X_flat = X.reshape(X.shape[0], -1)

# data split (80% train+val, 20% holdout)
X_train_val, X_holdout, y_train_val, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
# train-val split: test (80% of train+val aka is 64% of original) and val (20% of train+val aka is 16% of original)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_auc_scores = []
target_f1_scores = []
for train_index, test_index in skf.split(X_train, y_train):
    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
    y_train_fold, y_test_fold = y_train[train_index], y_train[test_index]
    n_minority_samples = min(np.bincount(y_train_fold))
    k_neighbors = max(1, min(5, n_minority_samples - 1))
    pipeline = IMBPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)),
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="linear", probability=True, random_state=42))
    ])
    pipeline.fit(X_train_fold, y_train_fold)
    y_pred = pipeline.predict(X_test_fold)
    y_pred_proba = pipeline.predict_proba(X_test_fold)[:, 1]
    roc_auc = roc_auc_score(y_test_fold, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test_fold, y_pred, pos_label=1)
    target_f1_scores.append(f1)
pipeline.fit(X_train_val, y_train_val)
y_holdout_pred = pipeline.predict(X_holdout)
y_holdout_pred_proba = pipeline.predict_proba(X_holdout)[:, 1]

print("classification report on holdout set - simple baseline:\n", classification_report(y_holdout, y_holdout_pred, target_names=["Non-Target", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_holdout_pred_proba)
print(f"holdout set roc auc: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"roc curve (auc = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

+- 100 ms baseline correction applied:

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)

header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
tmin, tmax = -0.2, 0.8 # time window around event
baseline = (-0.1, 0.1)  # baseline from -100 ms to +100 ms relative to the trigger
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=tmin, tmax=tmax, preload=True, baseline=baseline)
X = epochs.get_data()
y = epochs.events[:, -1] - 1
X_flat = X.reshape(X.shape[0], -1)

X_train_val, X_holdout, y_train_val, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_auc_scores = []
target_f1_scores = []
for train_index, test_index in skf.split(X_train, y_train):
    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
    y_train_fold, y_test_fold = y_train[train_index], y_train[test_index]
    n_minority_samples = min(np.bincount(y_train_fold))
    k_neighbors = max(1, min(5, n_minority_samples - 1))
    pipeline = IMBPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)),
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="linear", probability=True, random_state=42))
    ])
    pipeline.fit(X_train_fold, y_train_fold)
    y_pred = pipeline.predict(X_test_fold)
    y_pred_proba = pipeline.predict_proba(X_test_fold)[:, 1]
    roc_auc = roc_auc_score(y_test_fold, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test_fold, y_pred, pos_label=1)
    target_f1_scores.append(f1)

pipeline.fit(X_train_val, y_train_val)
y_holdout_pred = pipeline.predict(X_holdout)
y_holdout_pred_proba = pipeline.predict_proba(X_holdout)[:, 1]
print("classification report on holdout set - +- 100 ms baseline:\n", classification_report(y_holdout, y_holdout_pred, target_names=["Non-Target", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_holdout_pred_proba)
print(f"holdout set roc auc: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"roc curve (auc = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

## 3.MDM

The parameters for the MDM are the same as the ones chosen by the dataset's researchers: https://github.com/plcrodrigues/py.BI.EEG.2015a-GIPSA <br>

Baseline Model (no predictions):

In [ ]:
import pandas as pd
import numpy as np
import mne
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_paths, header_path):
    all_data = []
    all_labels = []
    scaler = StandardScaler() 

    for path in file_paths:
        header = pd.read_csv(header_path)
        data = pd.read_csv(path, names=header.columns)
        data = data.dropna()
        eeg_data = data.iloc[:, 1:33].values.T
        trigger_data = data["Trigger"].values
        target_data = data["Target"].values
        ch_names = list(header.columns[1:33])
        info = mne.create_info(ch_names=ch_names, sfreq=512, ch_types=["eeg"] * 32)
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
        events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
        event_id = {"NonTarget": 1, "Target": 2}
        epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)

        X = epochs.get_data()
        X = X.reshape(X.shape[0], -1)
        X = scaler.fit_transform(X) 
        X = X.reshape(-1, epochs.get_data().shape[1], epochs.get_data().shape[2])
        y = epochs.events[:, -1] - 1
        all_data.append(X)
        all_labels.extend(y)
    X_all = np.concatenate(all_data)
    y_all = np.array(all_labels)
    return X_all, y_all

header_path = str(EEG_DATA_DIR / 'Header.csv')
file_paths =[
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

X, y = load_and_preprocess_data(file_paths, header_path)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X.reshape(X.shape[0], -1), y)
X_resampled = X_resampled.reshape(-1, X.shape[1], X.shape[2])
X_train, X_holdout, y_train, y_holdout = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# baseline predictions
y_baseline_pred = np.zeros_like(y_holdout)  # everything as non-target class
print("classification report for baseline model:\n")
print(classification_report(y_holdout, y_baseline_pred, target_names=["NonTarget", "Target"]))

Multiple files, no baseline correction w/o normalization:

In [ ]:
import pandas as pd
import numpy as np
import mne
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from pyriemann.estimation import ERPCovariances
from pyriemann.classification import MDM

def load_and_preprocess_data(file_paths, header_path):
    all_data = []
    all_labels = []

    for path in file_paths:
        header = pd.read_csv(header_path)
        data = pd.read_csv(path, names=header.columns)
        data = data.dropna() 
        eeg_data = data.iloc[:, 1:33].values.T
        trigger_data = data["Trigger"].values
        target_data = data["Target"].values
        ch_names = list(header.columns[1:33])
        info = mne.create_info(ch_names=ch_names, sfreq=512, ch_types=["eeg"] * 32)
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
        events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
        event_id = {"NonTarget": 1, "Target": 2}
        epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
        X = epochs.get_data() 
        y = epochs.events[:, -1] - 1
        all_data.append(X)
        all_labels.extend(y)
    X_all = np.concatenate(all_data)
    y_all = np.array(all_labels)
    return X_all, y_all

header_path = str(EEG_DATA_DIR / 'Header.csv')
file_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

X, y = load_and_preprocess_data(file_paths, header_path)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X.reshape(X.shape[0], -1), y)
X_resampled = X_resampled.reshape(-1, X.shape[1], X.shape[2])
X_train, X_holdout, y_train, y_holdout = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ("erpcov", ERPCovariances(estimator="lwf")), 
    ("mdm", MDM())
])
param_grid = {
    "erpcov__estimator": ["lwf"],
    "mdm__metric": ["riemann"]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring="accuracy", n_jobs=-1) 
grid_search.fit(X_train, y_train)
print("best parameters:", grid_search.best_params_)
print("best cross-validation score:", grid_search.best_score_)
y_pred_holdout = grid_search.predict(X_holdout)
print("classification report for the holdout set:\n", classification_report(y_holdout, y_pred_holdout, target_names=["nontarget", "target"]))


Multiple files, with normalization, no baseline:

In [ ]:
def load_and_preprocess_data(file_paths, header_path):
    all_data = []
    all_labels = []
    scaler = StandardScaler()

    for path in file_paths:
        header = pd.read_csv(header_path)
        data = pd.read_csv(path, names=header.columns)
        data = data.dropna() 
        eeg_data = data.iloc[:, 1:33].values.T
        trigger_data = data["Trigger"].values
        target_data = data["Target"].values
        ch_names = list(header.columns[1:33])
        info = mne.create_info(ch_names=ch_names, sfreq=512, ch_types=["eeg"] * 32)
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
        events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
        event_id = {"NonTarget": 1, "Target": 2}
        epochs = Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
        X = epochs.get_data()
        X = X.reshape(X.shape[0], -1)
        X = scaler.fit_transform(X)
        X = X.reshape(-1, epochs.get_data().shape[1], epochs.get_data().shape[2])
        y = epochs.events[:, -1] - 1
        all_data.append(X)
        all_labels.extend(y)

    X_all = np.concatenate(all_data)
    y_all = np.array(all_labels)
    return X_all, y_all

header_path = str(EEG_DATA_DIR / 'Header.csv')
file_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

X, y = load_and_preprocess_data(file_paths, header_path)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X.reshape(X.shape[0], -1), y)
X_resampled = X_resampled.reshape(-1, X.shape[1], X.shape[2])
X_train_val, X_holdout, y_train_val, y_holdout = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)
pipeline = Pipeline([
    ("erpcov", ERPCovariances(estimator="lwf")), 
    ("mdm", MDM())
])

param_grid = {
    "erpcov__estimator": ["lwf"],
    "mdm__metric": ["riemann"]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)
print("best parameters:", grid_search.best_params_)
print("best cross-validation score:", grid_search.best_score_)
y_pred_holdout = grid_search.predict(X_holdout)
y_pred_holdout_proba = grid_search.predict_proba(X_holdout)[:, 1]
print("classification report for the holdout set - no baseline:\n", classification_report(y_holdout, y_pred_holdout, target_names=["NonTarget", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_pred_holdout_proba)
print(f"holdout set ROC AUC: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_pred_holdout_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"ROC curve (AUC = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

With baseline correction and normalization:

In [ ]:
def load_and_preprocess_data(file_paths, header_path):
    all_data = []
    all_labels = []
    scaler = StandardScaler()

    for path in file_paths:
        header = pd.read_csv(header_path)
        data = pd.read_csv(path, names=header.columns)
        data = data.dropna() 
        eeg_data = data.iloc[:, 1:33].values.T
        trigger_data = data["Trigger"].values
        target_data = data["Target"].values
        ch_names = list(header.columns[1:33])
        info = mne.create_info(ch_names=ch_names, sfreq=512, ch_types=["eeg"] * 32)
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
        events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
        event_id = {"NonTarget": 1, "Target": 2}
        epochs = Epochs(raw, events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True, baseline=(-0.2, 0))
        X = epochs.get_data()
        X = X.reshape(X.shape[0], -1)
        X = scaler.fit_transform(X)
        X = X.reshape(-1, epochs.get_data().shape[1], epochs.get_data().shape[2])
        y = epochs.events[:, -1] - 1
        all_data.append(X)
        all_labels.extend(y)
    X_all = np.concatenate(all_data)
    y_all = np.array(all_labels)
    return X_all, y_all

header_path = str(EEG_DATA_DIR / 'Header.csv')
file_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

X, y = load_and_preprocess_data(file_paths, header_path)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X.reshape(X.shape[0], -1), y)
X_resampled = X_resampled.reshape(-1, X.shape[1], X.shape[2])
X_train_val, X_holdout, y_train_val, y_holdout = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)
pipeline = Pipeline([
    ("erpcov", ERPCovariances(estimator="lwf")), 
    ("mdm", MDM())
])
param_grid = {
    "erpcov__estimator": ["lwf"],
    "mdm__metric": ["riemann"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("best parameters:", grid_search.best_params_)
print("best cross-validation score:", grid_search.best_score_)
y_pred_holdout = grid_search.predict(X_holdout)
y_pred_holdout_proba = grid_search.predict_proba(X_holdout)[:, 1]
print("classification report for the holdout set - simple baseline:\n", classification_report(y_holdout, y_pred_holdout, target_names=["NonTarget", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_pred_holdout_proba)
print(f"holdout set ROC AUC: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_pred_holdout_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"ROC curve (AUC = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

+- 100 ms baseline, with normalization:

In [ ]:
def load_and_preprocess_data(file_paths, header_path):
    all_data = []
    all_labels = []
    scaler = StandardScaler()

    for path in file_paths:
        header = pd.read_csv(header_path)
        data = pd.read_csv(path, names=header.columns)
        data = data.dropna() 
        eeg_data = data.iloc[:, 1:33].values.T
        trigger_data = data["Trigger"].values
        target_data = data["Target"].values
        ch_names = list(header.columns[1:33])
        info = mne.create_info(ch_names=ch_names, sfreq=512, ch_types=["eeg"] * 32)
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
        events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
        event_id = {"NonTarget": 1, "Target": 2}
        tmin, tmax = -0.2, 0.8 # time window around event
        baseline = (-0.1, 0.1)  # baseline from -100 ms to +100 ms relative to the trigger
        epochs = mne.Epochs(raw, events, event_id=event_id, tmin=tmin, tmax=tmax, preload=True, baseline=baseline)
        X = epochs.get_data()
        X = X.reshape(X.shape[0], -1)
        X = scaler.fit_transform(X)
        X = X.reshape(-1, epochs.get_data().shape[1], epochs.get_data().shape[2])
        y = epochs.events[:, -1] - 1
        all_data.append(X)
        all_labels.extend(y)
    X_all = np.concatenate(all_data)
    y_all = np.array(all_labels)
    return X_all, y_all

header_path = str(EEG_DATA_DIR / 'Header.csv')
file_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

X, y = load_and_preprocess_data(file_paths, header_path)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X.reshape(X.shape[0], -1), y)
X_resampled = X_resampled.reshape(-1, X.shape[1], X.shape[2])
X_train_val, X_holdout, y_train_val, y_holdout = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)
pipeline = Pipeline([
    ("erpcov", ERPCovariances(estimator="lwf")), 
    ("mdm", MDM())
])
param_grid = {
    "erpcov__estimator": ["lwf"],
    "mdm__metric": ["riemann"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)
print("best parameters:", grid_search.best_params_)
print("best cross-validation score:", grid_search.best_score_)
y_pred_holdout = grid_search.predict(X_holdout)
y_pred_holdout_proba = grid_search.predict_proba(X_holdout)[:, 1]
print("classification report for the holdout set - +- 100 ms baseline:\n", classification_report(y_holdout, y_pred_holdout, target_names=["NonTarget", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_pred_holdout_proba)
print(f"holdout set ROC AUC: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_pred_holdout_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"ROC curve (AUC = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

## 4. Random Forest

For the pipeline, i adapted the code here: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html <br>
This was used as a guide for hyperparameter tuning: https://www.geeksforgeeks.org/random-forest-hyperparameter-tuning-in-python/

Baseline model:

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)
header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
X = epochs.get_data()
y = epochs.events[:, -1] - 1
X_flat = X.reshape(X.shape[0], -1)
# apply z-score normalization
scaler = StandardScaler()
X_flat_scaled = scaler.fit_transform(X_flat)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_flat_scaled, y, test_size=0.2, random_state=42, stratify=y)

# baseline predictions(everything as the non-target class)
y_baseline_pred = np.zeros_like(y_holdout)
y_baseline_pred_proba = np.zeros_like(y_holdout, dtype=float) + 1
print("classification report for baseline model:\n")
print(classification_report(y_holdout, y_baseline_pred, target_names=["NonTarget", "Target"]))
roc_auc_baseline = roc_auc_score(y_holdout, y_baseline_pred_proba)
print(f"baseline model roc auc: {roc_auc_baseline:.4f}")
fpr_base, tpr_base, _ = roc_curve(y_holdout, y_baseline_pred_proba, pos_label=1)
plt.figure(figsize=(6, 6))
plt.plot(fpr_base, tpr_base, color="blue", lw=2, label=f"baseline roc curve (auc = {roc_auc_baseline:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - baseline model")
plt.legend(loc="lower right")
plt.show()

Evaluating on holdout set, no baseline:

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)

header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]
header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=0, tmax=0.8, preload=True, baseline=None)
X = epochs.get_data()
y = epochs.events[:, -1] - 1
X_flat = X.reshape(X.shape[0], -1)

X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_auc_scores = []
target_f1_scores = []
for train_index, test_index in skf.split(X_train_full, y_train_full):
    X_train, X_test = X_train_full[train_index], X_train_full[test_index]
    y_train, y_test = y_train_full[train_index], y_train_full[test_index]
    n_minority_samples = min(np.bincount(y_train))
    k_neighbors = max(1, min(5, n_minority_samples - 1))
    pipeline = IMBPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)),
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=500, # more estimators to reduce variance
            max_depth=15,  # set max depth to control overfitting
            min_samples_split=4, # 3 samples to be split
            min_samples_leaf=3, # 3 samples to be split for overfitting prevention
            max_features="sqrt", # default
            class_weight="balanced_subsample",  # here I used this due to class imbalance, it adjusts weights inversely proportional to class frequencies so minorirty class is more important
            random_state=42
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    target_f1_scores.append(f1)
pipeline.fit(X_train_full, y_train_full)
y_holdout_pred = pipeline.predict(X_holdout)
y_holdout_pred_proba = pipeline.predict_proba(X_holdout)[:, 1]

print("classification report on holdout set:\n", classification_report(y_holdout, y_holdout_pred, target_names=["Non-Target", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_holdout_pred_proba)
print(f"holdout set roc auc: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"roc curve (auc = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

With simple baseline correction:

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)

header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True, baseline=(-0.2, 0))
X = epochs.get_data()
y = epochs.events[:, -1] - 1

X_flat = X.reshape(X.shape[0], -1)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_auc_scores = []
target_f1_scores = []

for train_index, test_index in skf.split(X_train_full, y_train_full):
    X_train, X_test = X_train_full[train_index], X_train_full[test_index]
    y_train, y_test = y_train_full[train_index], y_train_full[test_index]
    
    n_minority_samples = min(np.bincount(y_train))
    k_neighbors = max(1, min(5, n_minority_samples - 1))
    
    pipeline = IMBPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)),
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=500, 
            max_depth=15,  
            min_samples_split=4, 
            min_samples_leaf=3, 
            max_features="sqrt",
            class_weight="balanced_subsample",  
            random_state=42
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    target_f1_scores.append(f1)
pipeline.fit(X_train_full, y_train_full)
y_holdout_pred = pipeline.predict(X_holdout)
y_holdout_pred_proba = pipeline.predict_proba(X_holdout)[:, 1]
print("classification report on holdout set:\n", classification_report(y_holdout, y_holdout_pred, target_names=["Non-Target", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_holdout_pred_proba)
print(f"holdout set roc auc: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"roc curve (auc = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()

With baseline correction (+-100 ms):

In [ ]:
warnings.simplefilter(action="ignore", category=FutureWarning)

header_path = str(EEG_DATA_DIR / 'Header.csv')
data_paths = [
    str(EEG_DATA_DIR / 'subject_02_csv/subject_02_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_03_csv/subject_03_session_03.csv'),
    str(EEG_DATA_DIR / 'subject_04_csv/subject_04_session_03.csv'),
]

header = pd.read_csv(header_path)
data_frames = [pd.read_csv(path, names=header.columns).dropna() for path in data_paths]
data = pd.concat(data_frames, ignore_index=True)
eeg_data = data.iloc[:, 1:33].values.T
trigger_data = data["Trigger"].values
target_data = data["Target"].values
info = mne.create_info(ch_names=list(header.columns[1:33]), sfreq=512, ch_types=["eeg"] * 32)
raw = mne.io.RawArray(eeg_data, info)
raw.filter(1, 24, l_trans_bandwidth="auto", h_trans_bandwidth="auto", filter_length="auto", fir_design="firwin")
events = np.column_stack((np.where(trigger_data == 1)[0], np.zeros_like(target_data[trigger_data == 1]), target_data[trigger_data == 1] + 1))
event_id = {"NonTarget": 1, "Target": 2}
tmin, tmax = -0.2, 0.8  # time window
baseline = (-0.1, 0.1)  # baseline interval from -100 ms to +100 ms from to the trigge
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=tmin, tmax=tmax, preload=True, baseline=baseline)
X = epochs.get_data()
y = epochs.events[:, -1] - 1
X_flat = X.reshape(X.shape[0], -1)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_auc_scores = []
target_f1_scores = []

for train_index, test_index in skf.split(X_train_full, y_train_full):
    X_train, X_test = X_train_full[train_index], X_train_full[test_index]
    y_train, y_test = y_train_full[train_index], y_train_full[test_index]
    n_minority_samples = min(np.bincount(y_train))
    k_neighbors = max(1, min(5, n_minority_samples - 1))
    pipeline = IMBPipeline([
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)),
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=500, 
            max_depth=15,  
            min_samples_split=4, 
            min_samples_leaf=3, 
            max_features="sqrt",
            class_weight="balanced_subsample",  
            random_state=42
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    roc_auc_scores.append(roc_auc)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    target_f1_scores.append(f1)
pipeline.fit(X_train_full, y_train_full)
y_holdout_pred = pipeline.predict(X_holdout)
y_holdout_pred_proba = pipeline.predict_proba(X_holdout)[:, 1]
print("classification report on holdout set:\n", classification_report(y_holdout, y_holdout_pred, target_names=["Non-Target", "Target"]))
roc_auc_holdout = roc_auc_score(y_holdout, y_holdout_pred_proba)
print(f"holdout set roc auc: {roc_auc_holdout:.4f}")
plt.figure(figsize=(6, 6))
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred_proba, pos_label=1)
plt.plot(fpr, tpr, color="green", lw=2, label=f"roc curve (auc = {roc_auc_holdout:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.title("receiver operating characteristic - holdout set")
plt.legend(loc="lower right")
plt.show()